# ¿Está bien asignado el esfuerzo comercial?

**Caso:** fuerza comercial en farma. 50 delegados, 2.000 médicos, dos años de histórico
(2024 y 2025) de visitas y prescripción a nivel médico-mes. Precio 18 €/unidad.

**Pregunta de Dirección Comercial:** ¿está bien asignado el esfuerzo? Si no, ¿dónde, qué
cambiar y cuánto vale el cambio? Con la misma plantilla y el mismo número de visitas.

## Resumen ejecutivo

1. **La regla actual se cumple a rajatabla:** los A reciben ~13 visitas al año, los B ~4 y
   los C ~1,5. La segmentación es buena (solo 30 de 2.000 médicos están en el segmento
   "equivocado" según su volumen real de categoría). El problema no es a quién se etiqueta
   como A, sino **cuántas veces se le visita**.
2. **La cuota responde a la visita, pero se satura.** Un médico sin visitas tiene ~12 % de
   cuota; con 3 a 5 visitas al año, ~21 %; a partir de 8-10 visitas la curva es plana
   (~23,5 %). La curva es la misma para A, B y C. Lo confirman los cambios dentro del
   mismo médico entre 2024 y 2025: la primera visita mueve ~3,6 puntos de cuota, la
   décima ~0,15 y la vigésima ~0,05.
3. **Por eso el esfuerzo está mal repartido:** el 22,5 % de las visitas de 2025 son la
   décima o posterior al mismo médico (casi todas a médicos A), mientras 1.000
   médicos-año (sobre todo B y C) no reciben ninguna. La décima visita a un A vale
   ~600 €; la quinta a un B, ~700 €; la segunda a un C, ~700 €. La primera visita a un B
   sin visitar vale más de 3.000 €.
4. **Recomendación:** pasar de 13 / 4 / 1,5 visitas al año a **≈ 9-10 / 5 / 2 (A / B / C)**,
   ajustando por volumen de categoría de cada médico, dentro de la cartera de cada
   delegado y sin mover una sola visita de un delegado a otro.
5. **Valor:** **≈ +2,4 M€ al año** (+8 % de ventas, cuota global de 19,9 % a ~21,5 %) a
   coste cero. Las variantes del modelo dan entre 2,25 y 2,5 M€; si solo se materializara
   la mitad del efecto estimado, 1,2 M€. Solo con "nadie a cero" ya se ganan ~1,3 M€.

El detalle, el método y los límites están en las secciones siguientes. Las cifras
finales se guardan en `output/resumen_cifras.json` y el plan por delegado y por médico
en `output/plan_visitas_2026.xlsx`.

## 0. Preparación

Decisiones de partida (ver también la sección 8, Límites y supuestos):

- **Duplicados.** `visitas.csv` tiene 255 filas con el mismo delegado, médico y fecha
  (253 pares y un triple, con `id_visita` distintos). Se tratan como doble registro del
  sistema y se eliminan. La sensibilidad a esta decisión se mide en la sección 5.
- **Grano de análisis: médico-año.** La fuerza comercial decide frecuencias anuales y
  los efectos de la visita duran meses (sección 3.3), así que el año es la unidad natural.
- **Potencial = volumen real de categoría del médico** (`unidades_categoria`), no la
  etiqueta A/B/C. La etiqueta sirve para comunicar; el volumen, para calcular.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import optimize

try:
    display
except NameError:  # ejecución como script
    display = print

ROOT = Path.cwd()
DATA, OUT = ROOT / "2_datos", ROOT / "output"
assert DATA.exists(), "Ejecuta desde la raíz del repositorio (donde está 2_datos/)"
OUT.mkdir(exist_ok=True)

PRECIO = 18.0  # €/unidad, constante en todo el periodo (enunciado)
COSTE_VISITA = 72.0  # € coste medio totalmente cargado (enunciado)

# Paleta: gris para "hoy", azul para "plan", naranja para el coste. Segmentos en rampa azul.
C_ACTUAL, C_PLAN, C_ACENTO = "#898781", "#2a78d6", "#eb6834"
C_TINTA, C_TINTA2, C_GRID = "#0b0b0b", "#52514e", "#e1e0d9"
C_SEG = {"A": "#1c5cab", "B": "#3987e5", "C": "#86b6ef"}
plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Segoe UI", "Arial", "DejaVu Sans"],
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#c3c2b7",
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": C_GRID, "grid.linewidth": 0.8,
    "axes.titlesize": 12, "axes.titleweight": "semibold", "axes.titlelocation": "left",
    "axes.labelcolor": C_TINTA2, "xtick.color": C_TINTA2, "ytick.color": C_TINTA2,
    "figure.dpi": 110, "savefig.dpi": 200, "legend.frameon": False, "figure.figsize": (8, 4.5),
})
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


def eur(x, dec=2):
    """Formato en millones de euros."""
    return f"{x / 1e6:,.{dec}f} M€"


def guardar(fig, nombre):
    fig.tight_layout()
    fig.savefig(OUT / f"{nombre}.png", bbox_inches="tight")
    if "get_ipython" in globals():  # en el notebook se muestra; como script solo se guarda
        plt.show()
    else:
        plt.close(fig)

In [2]:
med = pd.read_csv(DATA / "maestro_medicos.csv")
dele = pd.read_csv(DATA / "maestro_delegados.csv")
vis_raw = pd.read_csv(DATA / "visitas.csv", parse_dates=["fecha"])
pre = pd.read_csv(DATA / "prescripciones.csv")

# Comprobaciones de integridad (el enunciado la garantiza; se verifica igualmente)
assert len(med) == 2000 and len(dele) == 50 and len(vis_raw) == 16917 and len(pre) == 48000
assert vis_raw.id_medico.isin(med.id_medico).all() and vis_raw.id_delegado.isin(dele.id_delegado).all()
assert med.id_delegado_asignado.isin(dele.id_delegado).all()
assert (pre.unidades_producto <= pre.unidades_categoria).all() and (pre.unidades_categoria > 0).all()
assert pre.groupby("id_medico").size().eq(24).all()
assert not pre.duplicated(["id_medico", "mes"]).any()
# Cada visita la hace el delegado asignado al médico
assert (vis_raw.merge(med, on="id_medico").eval("id_delegado == id_delegado_asignado")).all()

# Duplicados: mismo delegado, mismo médico, mismo día
dup = vis_raw.duplicated(["id_delegado", "id_medico", "fecha"])
print(f"Visitas registradas: {len(vis_raw):,} | duplicadas mismo día: {dup.sum()} | válidas: {(~dup).sum():,}")
vis = vis_raw[~dup].copy()
vis["anio"] = vis.fecha.dt.year
pre["anio"] = pre.mes.str[:4].astype(int)

# Anclas de negocio
ventas = pre.groupby("anio").unidades_producto.sum() * PRECIO
cuota_global = pre.groupby("anio").unidades_producto.sum() / pre.groupby("anio").unidades_categoria.sum()
print("Ventas:", {a: eur(v) for a, v in ventas.items()}, "| Cuota global:", cuota_global.round(4).to_dict())

Visitas registradas: 16,917 | duplicadas mismo día: 255 | válidas: 16,662
Ventas: {2024: '30.27 M€', 2025: '30.61 M€'} | Cuota global: {2024: 0.1994, 2025: 0.1992}


### Tabla médico-año
Una fila por médico y año: visitas recibidas, unidades de categoría y de producto, cuota.

In [3]:
py = pre.groupby(["id_medico", "anio"], as_index=False)[["unidades_categoria", "unidades_producto"]].sum()
vy = vis.groupby(["id_medico", "anio"]).size().rename("visitas").reset_index()
dy = py.merge(vy, on=["id_medico", "anio"], how="left").fillna({"visitas": 0}).merge(med, on="id_medico")
dy["visitas"] = dy.visitas.astype(int)
dy["cuota"] = dy.unidades_producto / dy.unidades_categoria
assert len(dy) == 4000
dy.head()

,id_medico,anio,unidades_categoria,unidades_producto,visitas,segmento_potencial,region,brick,id_delegado_asignado,cuota
0,MED0001,2024,2077,514,8,C,Noreste,NOR-20,DEL041,0.247
1,MED0001,2025,2196,530,5,C,Noreste,NOR-20,DEL041,0.241
2,MED0002,2024,6086,973,1,B,Centro,CEN-19,DEL008,0.160
3,MED0002,2025,6073,784,0,B,Centro,CEN-19,DEL008,0.129
4,MED0003,2024,1706,327,2,C,Levante,LEV-17,DEL039,0.192


## 1. Cómo se reparte hoy el esfuerzo

La regla "los A se visitan más que los B y los B más que los C" se cumple, y con
intensidad: un A recibe 9 veces más visitas que un C. La pregunta es si esa intensidad
está justificada por la respuesta del médico, no por su potencial.

In [4]:
def reparto(df):
    g = df.groupby("segmento_potencial").agg(
        medicos=("id_medico", "size"), visitas=("visitas", "sum"),
        categoria=("unidades_categoria", "sum"), producto=("unidades_producto", "sum"),
    )
    g["visitas_por_medico"] = g.visitas / g.medicos
    g["pct_visitas"] = g.visitas / g.visitas.sum()
    g["pct_categoria"] = g.categoria / g.categoria.sum()
    g["cuota"] = g.producto / g.categoria
    g["sin_visitas"] = df[df.visitas == 0].groupby("segmento_potencial").size()
    return g


reparto_25 = reparto(dy[dy.anio == 2025])
display(reparto_25.round(3))
print("Médicos-año con 0 visitas (2024+2025):", (dy.visitas == 0).sum(),
      "| médicos sin ninguna visita en dos años:", (dy.groupby("id_medico").visitas.sum() == 0).sum())

,medicos,visitas,categoria,producto,visitas_por_medico,pct_visitas,pct_categoria,cuota,sin_visitas
segmento_potencial,,,,,,,,,
A,286,3806,3438146,761637,13.308,0.460,0.403,0.222,6
B,734,3023,3488125,667693,4.119,0.365,0.409,0.191,107
C,980,1449,1607725,270955,1.479,0.175,0.188,0.169,396


Médicos-año con 0 visitas (2024+2025): 1006 | médicos sin ninguna visita en dos años: 344


In [5]:
# ¿Está bien puesta la etiqueta A/B/C? Comparación con el ranking real por volumen de categoría
d25 = dy[dy.anio == 2025].sort_values("unidades_categoria", ascending=False).reset_index(drop=True)
nA, nB = (med.segmento_potencial == "A").sum(), (med.segmento_potencial == "B").sum()
d25["segmento_por_volumen"] = np.where(d25.index < nA, "A", np.where(d25.index < nA + nB, "B", "C"))
tabla_seg = pd.crosstab(d25.segmento_potencial, d25.segmento_por_volumen, margins=True)
display(tabla_seg)
mal_segmentados = int((d25.segmento_potencial != d25.segmento_por_volumen).sum())
print(f"Médicos cuya etiqueta no coincide con su volumen real: {mal_segmentados} de 2.000. "
      "La segmentación no es el problema.")

segmento_por_volumen,A,B,C,All
segmento_potencial,,,,
A,276,10,0,286
B,10,719,5,734
C,0,5,975,980
All,286,734,980,2000


Médicos cuya etiqueta no coincide con su volumen real: 30 de 2.000. La segmentación no es el problema.


In [6]:
# Distribución de visitas por médico en 2025: los extremos son el problema
bins = [-1, 0, 2, 5, 9, 14, 19, 29, 100]
etiquetas = ["0", "1-2", "3-5", "6-9", "10-14", "15-19", "20-29", "30+"]
dy["tramo"] = pd.cut(dy.visitas, bins=bins, labels=etiquetas)
dist = pd.crosstab(dy[dy.anio == 2025].tramo, dy[dy.anio == 2025].segmento_potencial)

fig, ax = plt.subplots()
abajo = np.zeros(len(dist))
for s in ["A", "B", "C"]:
    ax.bar(dist.index.astype(str), dist[s], bottom=abajo, color=C_SEG[s], label=f"Segmento {s}", width=0.7)
    abajo += dist[s].to_numpy()
for i, tot in enumerate(abajo):
    ax.text(i, tot + 8, f"{int(tot)}", ha="center", va="bottom", fontsize=9, color=C_TINTA2)
ax.set_title("Médicos por número de visitas recibidas en 2025")
ax.set_xlabel("Visitas al año"); ax.set_ylabel("Médicos"); ax.legend()
guardar(fig, "01_distribucion_visitas")

visitas_10mas = int((dy[dy.anio == 2025].visitas - 9).clip(lower=0).sum())
presupuesto_25 = int(dy[dy.anio == 2025].visitas.sum())
print(f"Visitas 2025 (sin duplicados): {presupuesto_25:,}. De ellas, {visitas_10mas:,} "
      f"({visitas_10mas / presupuesto_25:.0%}) son la décima o posterior al mismo médico.")

Visitas 2025 (sin duplicados): 8,278. De ellas, 1,864 (23%) son la décima o posterior al mismo médico.


C:\Users\Ricardo\AppData\Local\Temp\ipykernel_2032\1523713302.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. ¿Responde la prescripción a la visita?

### 2.1 Corte transversal: cuota según visitas recibidas, por segmento

A igual número de visitas, la cuota es prácticamente la misma sea el médico A, B o C.
Y la curva se aplana claramente a partir de 8-10 visitas al año.

In [7]:
curva_obs = (dy.groupby(["segmento_potencial", "tramo"], observed=True)
             .agg(n=("cuota", "size"), visitas=("visitas", "mean"), cuota=("cuota", "mean")))
display(curva_obs.cuota.unstack(0).round(3))
display(curva_obs.n.unstack(0))

segmento_potencial,A,B,C
tramo,,,
0,0.116,0.124,0.124
1-2,0.162,0.164,0.184
3-5,0.195,0.205,0.225
6-9,0.219,0.227,0.245
10-14,0.230,0.244,0.246
15-19,0.232,0.243,0.251
20-29,0.235,0.246,NaN


segmento_potencial,A,B,C
tramo,,,
0,10.000,213.000,783.000
1-2,22.000,316.000,821.000
3-5,52.000,551.000,269.000
6-9,67.000,282.000,59.000
10-14,146.000,68.000,24.000
15-19,162.000,25.000,4.000
20-29,113.000,13.000,NaN


### 2.2 Cambios dentro del mismo médico (primeras diferencias 2024 → 2025)

El corte transversal podría estar sesgado (quizá se visita más a quien ya prescribía más).
Por eso la estimación central usa **cambios dentro del mismo médico**: cuando a un médico
le suben o bajan las visitas de un año al otro, ¿cambia su cuota? Esto elimina cualquier
diferencia fija entre médicos. El resultado es el mismo: la respuesta es fuerte al
principio y casi nula a partir de 10 visitas.

In [8]:
W = dy.pivot(index="id_medico", columns="anio", values=["visitas", "cuota", "unidades_categoria"])
v24, v25 = W["visitas"][2024].to_numpy(float), W["visitas"][2025].to_numpy(float)
dcuota = (W["cuota"][2025] - W["cuota"][2024]).to_numpy()
dvis = v25 - v24
seg_w = med.set_index("id_medico").loc[W.index, "segmento_potencial"]

nivel = pd.cut(v24, [-1, 0, 2, 5, 9, 14, 19, 100], labels=["0", "1-2", "3-5", "6-9", "10-14", "15-19", "20+"])
filas = []
for niv in nivel.categories:
    m = nivel == niv
    if m.sum() < 30 or dvis[m].std() == 0:
        continue
    ols = sm.OLS(dcuota[m], sm.add_constant(dvis[m])).fit(cov_type="HC1")
    filas.append({"visitas_2024": niv, "medicos": int(m.sum()), "pp_cuota_por_visita": 100 * ols.params[1],
                  "t": ols.tvalues[1], "visitas_medias": v24[m].mean()})
fd_niveles = pd.DataFrame(filas).set_index("visitas_2024")
display(fd_niveles.round(2))

fig, ax = plt.subplots()
ax.bar(fd_niveles.index.astype(str), fd_niveles.pp_cuota_por_visita, color=C_PLAN, width=0.65)
for i, v in enumerate(fd_niveles.pp_cuota_por_visita):
    ax.text(i, v + 0.05, f"{v:.2f}", ha="center", va="bottom", fontsize=9, color=C_TINTA2)
ax.set_title("Puntos de cuota que gana el médico por cada visita adicional, según cuántas ya recibía")
ax.set_xlabel("Visitas recibidas en 2024"); ax.set_ylabel("Puntos porcentuales de cuota por visita")
guardar(fig, "02_respuesta_marginal")

,medicos,pp_cuota_por_visita,t,visitas_medias
visitas_2024,,,,
0,497,3.650,26.070,0.000
1-2,588,2.410,22.770,1.400
3-5,437,0.940,19.700,3.860
6-9,199,0.330,9.800,7.080
10-14,116,0.140,6.750,11.980
15-19,95,0.080,4.170,16.790
20+,68,0.050,2.420,21.690


C:\Users\Ricardo\AppData\Local\Temp\ipykernel_2032\1523713302.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 2.3 ¿Cuánto tarda el efecto? (panel mensual, efectos fijos por médico)

Una visita mueve la cuota desde el mismo mes y el efecto sigue vivo seis meses después.
Por eso el análisis anual captura el efecto casi completo dentro del mismo año.

In [9]:
vm = vis.groupby(["id_medico", vis.fecha.dt.strftime("%Y-%m")]).size().rename("v").reset_index().rename(columns={"fecha": "mes"})
pm = pre.merge(vm, on=["id_medico", "mes"], how="left").fillna({"v": 0}).sort_values(["id_medico", "mes"])
pm["cuota"] = pm.unidades_producto / pm.unidades_categoria
LAGS = [f"v_lag{k}" for k in range(7)]
for k in range(7):
    pm[f"v_lag{k}"] = pm.groupby("id_medico").v.shift(k)
pm_ok = pm.dropna(subset=LAGS)
dentro = pm_ok[["cuota"] + LAGS] - pm_ok.groupby("id_medico")[["cuota"] + LAGS].transform("mean")
ols_m = sm.OLS(dentro.cuota, dentro[LAGS]).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(pm_ok.id_medico)[0]})
efecto_mensual = pd.DataFrame({"pp_cuota": 100 * ols_m.params, "t": ols_m.tvalues}).round(2)
efecto_mensual.index = [f"mes t-{k}" for k in range(7)]
display(efecto_mensual)

# ¿Hay selección? Médicos sin visitas en 2024 que empiezan a recibirlas en 2025: su cuota
# ANTES de la primera visita, frente a la de los médicos que siguen sin visitas.
primera_25 = vis[vis.anio == 2025].groupby("id_medico").fecha.min()
nuevos = primera_25.index.difference(vis[vis.anio == 2024].id_medico.unique())
antes = pm[pm.id_medico.isin(nuevos)].copy()
antes["mes_dt"] = pd.to_datetime(antes.mes)
antes = antes[(antes.mes_dt < antes.id_medico.map(primera_25)) & (antes.mes_dt >= antes.id_medico.map(primera_25) - pd.DateOffset(months=6))]
nunca = pm[~pm.id_medico.isin(vis.id_medico.unique())]
print(f"Médicos que pasan de 0 visitas en 2024 a alguna en 2025: {len(nuevos)}. Su cuota en los 6 meses previos a la "
      f"primera visita: {antes.cuota.mean():.1%}; cuota de los médicos nunca visitados: {nunca.cuota.mean():.1%}.")

,pp_cuota,t
mes t-0,0.250,12.680
mes t-1,0.240,11.730
mes t-2,0.220,10.450
mes t-3,0.210,9.700
mes t-4,0.140,6.910
mes t-5,0.170,8.580
mes t-6,0.130,6.940


Médicos que pasan de 0 visitas en 2024 a alguna en 2025: 153. Su cuota en los 6 meses previos a la primera visita: 16.4%; cuota de los médicos nunca visitados: 12.4%.


### 2.4 La curva de respuesta

Ajustamos una curva de saturación sencilla, `cuota = base_médico + D · v / (v + h)`, donde `v`
son las visitas al año. `D` es la ganancia máxima de cuota que aporta la visita y `h` es el
número de visitas con el que se alcanza la mitad de esa ganancia. Se estima **sobre los cambios
dentro del mismo médico** (primeras diferencias), no sobre niveles, y se comprueba que la
misma curva sirve para A, B y C.

In [10]:
def g(v, D, h):
    """Ganancia de cuota (en tanto por uno) por recibir v visitas al año."""
    return D * v / (v + h)


def ajustar_fd(v_a, v_b, dc, p0=(0.15, 2.5)):
    """Ajuste por mínimos cuadrados de D y h sobre primeras diferencias: dc = g(v_b) - g(v_a)."""
    res = optimize.least_squares(lambda p: g(v_b, *p) - g(v_a, *p) - dc, p0, bounds=([0, 0.05], [1, 60]))
    return float(res.x[0]), float(res.x[1])


D, h = ajustar_fd(v24, v25, dcuota)
V_all, C_all = dy.visitas.to_numpy(float), dy.cuota.to_numpy()
s0 = float(np.mean(C_all - g(V_all, D, h)))  # cuota base media (0 visitas)
print(f"Curva (primeras diferencias): D = {D:.4f}, h = {h:.2f} | cuota base media = {s0:.3f} | techo = {s0 + D:.3f}")

# Robustez: la misma curva por segmento y sobre niveles
ajustes = {}
for s in ["A", "B", "C"]:
    m = (seg_w == s).to_numpy()
    ajustes[f"FD segmento {s}"] = ajustar_fd(v24[m], v25[m], dcuota[m])
res_niv = optimize.least_squares(lambda p: p[0] + g(V_all, p[1], p[2]) - C_all, [0.12, 0.13, 1.7], bounds=([0, 0, 0.05], [1, 1, 60]))
ajustes["Niveles (todos)"] = (float(res_niv.x[1]), float(res_niv.x[2]))
ajustes["FD todos (central)"] = (D, h)
tabla_ajustes = pd.DataFrame(ajustes, index=["D", "h"]).T
tabla_ajustes["cuota extra con 5 visitas"] = tabla_ajustes.apply(lambda r: g(5, r.D, r.h), axis=1)
tabla_ajustes["cuota extra con 15 visitas"] = tabla_ajustes.apply(lambda r: g(15, r.D, r.h), axis=1)
display(tabla_ajustes.round(3))

# Pendiente del modelo vs pendiente observada por nivel (validación)
vm_ = fd_niveles.visitas_medias
fd_niveles["pp_modelo_una_visita_mas"] = 100 * (g(vm_ + 1, D, h) - g(vm_, D, h))
display(fd_niveles[["medicos", "pp_cuota_por_visita", "pp_modelo_una_visita_mas"]].round(2))

Curva (primeras diferencias): D = 0.1590, h = 2.54 | cuota base media = 0.117 | techo = 0.276


,D,h,cuota extra con 5 visitas,cuota extra con 15 visitas
FD segmento A,0.115,2.844,0.073,0.097
FD segmento B,0.154,3.840,0.087,0.123
FD segmento C,0.167,2.370,0.113,0.144
Niveles (todos),0.127,1.725,0.095,0.114
FD todos (central),0.159,2.536,0.106,0.136


,medicos,pp_cuota_por_visita,pp_modelo_una_visita_mas
visitas_2024,,,
0,497,3.650,4.500
1-2,588,2.410,2.070
3-5,437,0.940,0.850
6-9,199,0.330,0.400
10-14,116,0.140,0.180
15-19,95,0.080,0.100
20+,68,0.050,0.070


In [11]:
fig, ax = plt.subplots()
vv = np.linspace(0, 30, 200)
ax.plot(vv, 100 * (s0 + g(vv, D, h)), color=C_TINTA, lw=2, label="Curva usada para valorar (cambios dentro del médico)")
ax.plot(vv, 100 * (res_niv.x[0] + g(vv, *ajustes["Niveles (todos)"])), color=C_ACTUAL, lw=1.5, ls="--", label="Ajuste sobre niveles (referencia)")
for s in ["A", "B", "C"]:
    c = curva_obs.loc[s]
    ax.scatter(c.visitas, 100 * c.cuota, s=np.sqrt(c.n) * 4, color=C_SEG[s], label=f"Observado segmento {s}", zorder=3, edgecolor="white", linewidth=0.5)
for s, x in reparto_25.visitas_por_medico.items():
    ax.axvline(x, color=C_SEG[s], lw=1, ls=":", ymax=0.92)
    ax.text(x, 100 * (s0 + D) + (2.2 if s != "B" else 1.2), f"hoy {s}: {x:.1f}", ha="center", fontsize=8.5, color=C_SEG[s])
ax.set_ylim(10, 30.5); ax.set_xlim(-0.5, 30)
ax.set_title("Cuota del médico según visitas recibidas al año (el tamaño del punto es el número de médicos)")
ax.set_xlabel("Visitas al año"); ax.set_ylabel("Cuota de nuestro producto (%)")
ax.legend(loc="lower right", fontsize=9)
guardar(fig, "03_curva_respuesta")

C:\Users\Ricardo\AppData\Local\Temp\ipykernel_2032\1523713302.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Cuánto vale cada visita, en euros

Valor de una visita = volumen de categoría del médico × ganancia de cuota × 18 €. Como la
ganancia de cuota depende de cuántas visitas ya recibe, la décima visita a un A vale mucho
menos que la primera a un B, aunque el A tenga el triple de volumen.

In [12]:
base = dy[dy.anio == 2025].set_index("id_medico")  # potencial (categoría) y asignación actual
cat, v_act, seg = base.unidades_categoria.astype(float), base.visitas.astype(float), base.segmento_potencial


def delta_unidades(v_new, v_old, D=D, h=h):
    return cat * (g(v_new, D, h) - g(v_old, D, h))


valor_siguiente = delta_unidades(v_act + 1, v_act) * PRECIO
valor_ultima = delta_unidades(v_act, (v_act - 1).clip(lower=0)) * PRECIO
tabla_valor = pd.DataFrame({
    "visitas_por_medico": v_act.groupby(seg).mean(),
    "categoria_media": cat.groupby(seg).mean(),
    "eur_una_visita_mas": valor_siguiente.groupby(seg).mean(),
    "eur_ultima_visita": valor_ultima[v_act > 0].groupby(seg).mean(),
})
display(tabla_valor.round(0))

# Valor de la visita k-ésima para el médico mediano de cada segmento
ks = np.arange(1, 21)
cat_mediana = cat.groupby(seg).median()
fig, ax = plt.subplots()
for s in ["A", "B", "C"]:
    val = cat_mediana[s] * (g(ks, D, h) - g(ks - 1, D, h)) * PRECIO
    ax.plot(ks, val, color=C_SEG[s], lw=2, marker="o", ms=4, label=f"Médico {s} mediano ({cat_mediana[s]:,.0f} uds. de categoría al año)".replace(",", "."))
    ax.text(ks[-1] + 0.3, val[-1], s, color=C_SEG[s], va="center", fontsize=9)
ax.axhline(COSTE_VISITA, color=C_ACENTO, lw=1.2, ls="--")
ax.text(20.5, COSTE_VISITA + 60, f"coste de la visita: {COSTE_VISITA:.0f} €", color=C_ACENTO, ha="right", fontsize=9)
ax.set_yscale("log"); ax.set_yticks([50, 100, 300, 1000, 3000, 10000]); ax.set_yticklabels(["50", "100", "300", "1.000", "3.000", "10.000"])
ax.set_xticks(range(1, 21))
ax.set_title("Euros de venta anual que aporta la visita número k a un médico (escala logarítmica)")
ax.set_xlabel("Número de visita en el año"); ax.set_ylabel("€ por visita"); ax.legend(fontsize=9)
guardar(fig, "04_valor_visita_k")

k_val = pd.DataFrame({s: cat_mediana[s] * (g(ks, D, h) - g(ks - 1, D, h)) * PRECIO for s in "ABC"}, index=ks).round(0)
display(k_val.loc[[1, 2, 3, 5, 8, 10, 14, 20]].T)

,visitas_por_medico,categoria_media,eur_una_visita_mas,eur_ultima_visita
segmento_potencial,,,,
A,13.000,"12,021.000",824.000,853.000
B,4.000,"4,752.000","1,298.000","1,333.000"
C,1.000,"1,641.000",844.000,855.000


C:\Users\Ricardo\AppData\Local\Temp\ipykernel_2032\1523713302.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,1,2,3,5,8,10,14,20
A,"9,881.000","5,524.000","3,528.000","1,799.000",882.000,613.000,345.000,183.000
B,"3,833.000","2,143.000","1,369.000",698.000,342.000,238.000,134.000,71.000
C,"1,332.000",745.000,476.000,243.000,119.000,83.000,46.000,25.000


## 4. La reasignación: mismas visitas, mejor repartidas

Con la curva estimada, el reparto que maximiza unidades es el que **iguala el valor de la
última visita** entre médicos: se asignan las visitas una a una a quien más aporta en ese
momento (asignación greedy, óptima porque la curva es cóncava).

Se calculan tres versiones:

- **Óptimo global:** una única bolsa de visitas para toda la compañía (cota superior).
- **Óptimo dentro de cada delegado:** cada delegado mantiene exactamente sus visitas de 2025
  y las reparte mejor entre sus propios médicos. **Es el plan recomendado**, porque es
  ejecutable el lunes: no cambia territorios, cargas ni costes.
- **Regla sencilla por segmento** (para comunicar): la misma frecuencia para todos los
  médicos de un segmento.

In [13]:
def asignacion_optima(cat, presupuesto, D=D, h=h, vmax=40):
    """Reparte `presupuesto` visitas entre médicos con volumen `cat` maximizando unidades."""
    ks = np.arange(1, vmax + 1)
    ganancia = np.outer(cat.to_numpy(float), g(ks, D, h) - g(ks - 1, D, h))  # ganancia de la visita k a cada médico
    elegidas = np.argsort(ganancia, axis=None)[::-1][: int(presupuesto)]
    return pd.Series(np.bincount(elegidas // vmax, minlength=len(cat)), index=cat.index)


def valor_plan(v_plan, D=D, h=h):
    """Ventas anuales adicionales (€) de una asignación frente a la actual, con la curva (D, h)."""
    return float(delta_unidades(v_plan, v_act, D, h).sum() * PRECIO)


unidades_25 = base.unidades_producto.sum()

# a) Óptimo global
v_global = asignacion_optima(cat, presupuesto_25)
# b) Óptimo dentro de cada delegado (plan recomendado)
v_plan = pd.Series(0, index=cat.index)
for _, ids in base.groupby("id_delegado_asignado").groups.items():
    v_plan.loc[ids] = asignacion_optima(cat.loc[ids], v_act.loc[ids].sum())
assert v_plan.sum() == presupuesto_25
assert (v_plan.groupby(base.id_delegado_asignado).sum() == v_act.groupby(base.id_delegado_asignado).sum()).all()
# c) Regla sencilla: frecuencia fija por segmento (redondeo del plan), sin superar el presupuesto
regla = v_plan.groupby(seg).mean().round().astype(int)
v_regla = seg.map(regla).astype(float)
while v_regla.sum() > presupuesto_25:  # si la regla se pasa del presupuesto, se recorta a los A
    regla["A"] -= 1; v_regla = seg.map(regla).astype(float)
# d) "Nadie a cero": dos visitas a cada médico sin visitar, quitadas a quien recibe más de 12
v_nadie0 = v_act.copy(); ceros = v_nadie0 == 0; v_nadie0[ceros] = 2
por_quitar = int(2 * ceros.sum())
while por_quitar > 0:
    i = v_nadie0.idxmax(); v_nadie0[i] -= 1; por_quitar -= 1

escenarios = pd.DataFrame({
    "Óptimo global": v_global, "Plan (óptimo por delegado)": v_plan,
    f"Regla {regla['A']}/{regla['B']}/{regla['C']}": v_regla, "Nadie a cero": v_nadie0, "Hoy (2025)": v_act,
})
resumen_esc = pd.DataFrame({
    "visitas": escenarios.sum(),
    "A": escenarios.groupby(seg).mean().loc["A"], "B": escenarios.groupby(seg).mean().loc["B"], "C": escenarios.groupby(seg).mean().loc["C"],
    "ventas_extra_M€": [valor_plan(escenarios[c]) / 1e6 for c in escenarios],
})
resumen_esc["pct_ventas"] = resumen_esc["ventas_extra_M€"] * 1e6 / ventas[2025]
resumen_esc["cuota_global"] = (unidades_25 + resumen_esc["ventas_extra_M€"] * 1e6 / PRECIO) / cat.sum()
display(resumen_esc.round(3))

,visitas,A,B,C,ventas_extra_M€,pct_ventas,cuota_global
Óptimo global,"8,278.000",9.476,5.023,1.919,2.498,0.082,0.216
Plan (óptimo por delegado),"8,278.000",9.556,4.995,1.917,2.441,0.080,0.215
Regla 9/5/2,"8,204.000",9.000,5.000,2.000,2.380,0.078,0.215
Nadie a cero,"8,278.000",10.213,4.270,2.268,1.305,0.043,0.208
Hoy (2025),"8,278.000",13.308,4.119,1.479,0.000,0.000,0.199


In [14]:
delta_eur = delta_unidades(v_plan, v_act) * PRECIO
valor_base = float(delta_eur.sum())
bridge = delta_eur.groupby(seg).sum()  # neto por segmento
perdida = float(delta_eur[v_plan < v_act].sum())  # lo que se pierde en los médicos a los que se quitan visitas
ganancia = delta_eur[v_plan > v_act].groupby(seg).sum()  # lo que se gana en los médicos que reciben más
n_bajan = (v_plan < v_act).groupby(seg).sum()
print("Plan recomendado:", eur(valor_base), f"({valor_base / ventas[2025]:+.1%} de ventas)",
      "| neto por segmento:", {s: eur(v) for s, v in bridge.items()})
print("Se pierde", eur(-perdida), f"en {int(n_bajan.sum())} médicos que reciben menos visitas (A: {n_bajan['A']}, B: {n_bajan['B']}, C: {n_bajan['C']});",
      "se gana", {s: eur(v) for s, v in ganancia.items()}, "en los que reciben más.")

fig, ax = plt.subplots(figsize=(8, 4.5))
pasos = [(f"Se pierde en los\n{int(n_bajan.sum())} médicos que\nreciben menos visitas", perdida),
         (f"Se gana en A\n({int((v_plan > v_act)[seg == 'A'].sum())} médicos)", ganancia["A"]),
         (f"Se gana en B\n({int((v_plan > v_act)[seg == 'B'].sum())} médicos)", ganancia["B"]),
         (f"Se gana en C\n({int((v_plan > v_act)[seg == 'C'].sum())} médicos)", ganancia["C"])]
acum = 0
for i, (nombre, val) in enumerate(pasos):
    ax.bar(i, val / 1e6, bottom=acum / 1e6, color=C_ACENTO if val < 0 else C_PLAN, width=0.6)
    y_txt = (acum + val) / 1e6 + 0.06 if val > 0 else acum / 1e6 + 0.06  # la pérdida se etiqueta sobre la línea de cero
    ax.text(i, y_txt, f"{val / 1e6:+.2f}", ha="center", va="bottom", fontsize=10, color=C_TINTA)
    acum += val
ax.bar(4, acum / 1e6, color=C_TINTA, width=0.6)
ax.text(4, acum / 1e6 + 0.06, f"{acum / 1e6:+.2f} M€", ha="center", va="bottom", fontsize=11, fontweight="semibold", color=C_TINTA)
ax.set_xticks(range(5)); ax.set_xticklabels([p[0] for p in pasos] + ["Neto al año"], fontsize=9)
ax.set_ylim(min(perdida / 1e6, 0) - 0.2, acum / 1e6 + 0.4)
ax.axhline(0, color="#c3c2b7", lw=1)
ax.set_ylabel("Ventas adicionales (M€ / año)"); ax.set_title("De dónde sale el valor: mismas visitas, mejor repartidas")
guardar(fig, "05_puente_valor")

Plan recomendado: 2.44 M€ (+8.0% de ventas) | neto por segmento: {'A': '0.05 M€', 'B': '1.60 M€', 'C': '0.80 M€'}
Se pierde 0.92 M€ en 595 médicos que reciben menos visitas (A: 198, B: 205, C: 192); se gana {'A': '0.53 M€', 'B': '1.86 M€', 'C': '0.98 M€'} en los que reciben más.


C:\Users\Ricardo\AppData\Local\Temp\ipykernel_2032\1523713302.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# Cómo cambia la frecuencia por médico: hoy vs plan, por segmento
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(3); w = 0.36
hoy = v_act.groupby(seg).mean(); plan_m = v_plan.groupby(seg).mean()
ax.bar(x - w / 2, hoy, w, color=C_ACTUAL, label="Hoy (2025)")
ax.bar(x + w / 2, plan_m, w, color=C_PLAN, label="Plan")
for i, s in enumerate("ABC"):
    ax.text(i - w / 2, hoy[s] + 0.2, f"{hoy[s]:.1f}", ha="center", fontsize=10, color=C_TINTA2)
    ax.text(i + w / 2, plan_m[s] + 0.2, f"{plan_m[s]:.1f}", ha="center", fontsize=10, color=C_PLAN, fontweight="semibold")
    rango = f"{int(v_plan[seg == s].min())}-{int(v_plan[seg == s].max())}"
    ax.text(i + w / 2, -1.6, f"rango {rango}", ha="center", fontsize=8.5, color=C_TINTA2)
ax.set_xticks(x); ax.set_xticklabels([f"Segmento {s}\n({(seg == s).sum()} médicos)" for s in "ABC"])
ax.set_ylim(-2.2, 16); ax.set_ylabel("Visitas al año por médico"); ax.set_title("Frecuencia media por segmento: hoy y plan")
ax.legend()
guardar(fig, "06_frecuencia_hoy_vs_plan")

C:\Users\Ricardo\AppData\Local\Temp\ipykernel_2032\1523713302.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.1 Robustez: ¿cuánto depende el valor del modelo?

El plan se fija (el de arriba) y se vuelve a valorar con curvas alternativas: la ajustada
sobre niveles, una curva distinta por segmento, un remuestreo bootstrap de médicos,
la versión sin eliminar duplicados y un escenario en el que solo se materializa la mitad
del efecto estimado.

In [16]:
rng = np.random.default_rng(42)
n = len(v24)
boot = []
for _ in range(300):
    idx = rng.integers(0, n, n)
    Db, hb = ajustar_fd(v24[idx], v25[idx], dcuota[idx])
    boot.append(valor_plan(v_plan, Db, hb))
ic_boot = np.percentile(boot, [2.5, 97.5])

# Curva por segmento: cada médico se valora con la curva de su segmento
val_seg = sum(float((delta_unidades(v_plan, v_act, *ajustes[f"FD segmento {s}"])[seg == s]).sum()) for s in "ABC") * PRECIO

# Sin eliminar duplicados
vy_raw = vis_raw[vis_raw.fecha.dt.year == 2025].groupby("id_medico").size().reindex(cat.index, fill_value=0).astype(float)
v_plan_raw = pd.Series(0, index=cat.index)
for _, ids in base.groupby("id_delegado_asignado").groups.items():
    v_plan_raw.loc[ids] = asignacion_optima(cat.loc[ids], vy_raw.loc[ids].sum())
val_raw = float((cat * (g(v_plan_raw, D, h) - g(vy_raw, D, h))).sum() * PRECIO)

robustez = pd.DataFrame({
    "ventas_extra_M€": {
        "Central (FD, sin duplicados)": valor_base / 1e6,
        "Curva ajustada sobre niveles": valor_plan(v_plan, *ajustes["Niveles (todos)"]) / 1e6,
        "Curva distinta por segmento": val_seg / 1e6,
        "Bootstrap IC 95 % (inferior)": ic_boot[0] / 1e6,
        "Bootstrap IC 95 % (superior)": ic_boot[1] / 1e6,
        "Manteniendo los duplicados": val_raw / 1e6,
        "Solo se materializa el 50 % del efecto": valor_base / 2e6,
    }
})
display(robustez.round(2))
rango_prudente = (robustez["ventas_extra_M€"].min(), robustez["ventas_extra_M€"].max())

,ventas_extra_M€
"Central (FD, sin duplicados)",2.440
Curva ajustada sobre niveles,2.260
Curva distinta por segmento,2.250
Bootstrap IC 95 % (inferior),2.390
Bootstrap IC 95 % (superior),2.490
Manteniendo los duplicados,2.480
Solo se materializa el 50 % del efecto,1.220


## 5. El plan, médico a médico y delegado a delegado

Cada delegado conserva su número de visitas. Lo que cambia es a quién y cuántas veces.
El fichero `output/plan_visitas_2026.xlsx` tiene tres hojas: resumen por segmento, plan por
delegado y plan por médico (la target list con la frecuencia recomendada).

In [17]:
plan_med = pd.DataFrame({
    "id_delegado": base.id_delegado_asignado, "region": base.region, "segmento": seg,
    "categoria_2025": cat.astype(int), "cuota_2025": base.cuota.round(3),
    "visitas_2025": v_act.astype(int), "visitas_plan": v_plan.astype(int),
})
plan_med["delta_visitas"] = plan_med.visitas_plan - plan_med.visitas_2025
plan_med["delta_unidades"] = delta_unidades(v_plan, v_act).round(0).astype(int)
plan_med["delta_eur"] = (plan_med.delta_unidades * PRECIO).round(0).astype(int)
plan_med = plan_med.sort_values(["id_delegado", "segmento", "categoria_2025"], ascending=[True, True, False])

plan_del = plan_med.groupby("id_delegado").agg(
    region=("region", "first"), medicos=("segmento", "size"), visitas=("visitas_2025", "sum"),
    medicos_suben=("delta_visitas", lambda s: int((s > 0).sum())), medicos_bajan=("delta_visitas", lambda s: int((s < 0).sum())),
    visitas_movidas=("delta_visitas", lambda s: int(s.clip(lower=0).sum())),
    A_hoy=("visitas_2025", lambda s: s[plan_med.loc[s.index, "segmento"] == "A"].mean()),
    A_plan=("visitas_plan", lambda s: s[plan_med.loc[s.index, "segmento"] == "A"].mean()),
    B_hoy=("visitas_2025", lambda s: s[plan_med.loc[s.index, "segmento"] == "B"].mean()),
    B_plan=("visitas_plan", lambda s: s[plan_med.loc[s.index, "segmento"] == "B"].mean()),
    C_hoy=("visitas_2025", lambda s: s[plan_med.loc[s.index, "segmento"] == "C"].mean()),
    C_plan=("visitas_plan", lambda s: s[plan_med.loc[s.index, "segmento"] == "C"].mean()),
    ventas_extra_eur=("delta_eur", "sum"),
).join(dele.set_index("id_delegado")[["antiguedad_anos", "coste_visita_eur"]])
plan_del["pct_ventas_extra"] = plan_del.ventas_extra_eur / (base.groupby("id_delegado_asignado").unidades_producto.sum() * PRECIO)
display(plan_del.round(2).head(10))

plan_reg = plan_med.groupby("region").agg(medicos=("segmento", "size"), visitas=("visitas_2025", "sum"), ventas_extra_eur=("delta_eur", "sum"))
display(plan_reg)

resumen_plan = pd.DataFrame({"hoy": v_act.groupby(seg).mean(), "plan": v_plan.groupby(seg).mean(),
                             "plan_min": v_plan.groupby(seg).min(), "plan_max": v_plan.groupby(seg).max(),
                             "ventas_extra_eur": bridge}).round(2)
with pd.ExcelWriter(OUT / "plan_visitas_2026.xlsx", engine="openpyxl") as xw:
    resumen_plan.to_excel(xw, sheet_name="resumen_segmento")
    plan_del.round(2).to_excel(xw, sheet_name="plan_por_delegado")
    plan_med.to_excel(xw, sheet_name="plan_por_medico")
plan_med.to_csv(OUT / "plan_por_medico.csv")

suben, bajan = int((plan_med.delta_visitas > 0).sum()), int((plan_med.delta_visitas < 0).sum())
de_cero = int(((plan_med.visitas_2025 == 0) & (plan_med.visitas_plan > 0)).sum())
movidas = int(plan_med.delta_visitas.clip(lower=0).sum())
print(f"Médicos que suben: {suben} (de ellos {de_cero} pasan de 0 a alguna visita) | bajan: {bajan} | "
      f"visitas que cambian de médico: {movidas:,} de {presupuesto_25:,} ({movidas / presupuesto_25:.0%})")
print("Ningún delegado cambia su carga:", (plan_del.visitas == v_act.groupby(base.id_delegado_asignado).sum()).all())

,region,medicos,visitas,medicos_suben,medicos_bajan,visitas_movidas,A_hoy,A_plan,B_hoy,B_plan,C_hoy,C_plan,ventas_extra_eur,antiguedad_anos,coste_visita_eur,pct_ventas_extra
id_delegado,,,,,,,,,,,,,,,,
DEL001,Noreste,55,203,31,17,65,11.880,9.000,3.760,4.650,1.470,1.730,55476,13,61.470,0.070
DEL002,Noroeste,37,163,20,10,54,14.670,9.330,2.930,5.210,2.000,2.000,49626,13,78.830,0.080
DEL003,Centro,41,148,23,11,46,13.170,9.170,3.540,4.230,1.050,1.730,38574,10,69.510,0.060
DEL004,Levante,35,137,25,7,58,17.400,9.200,2.770,4.620,0.820,1.820,67194,15,67.510,0.130
DEL005,Sur,19,76,11,7,24,14.000,10.000,6.330,5.440,0.560,1.890,15336,1,75.850,0.050
DEL006,Noreste,50,245,30,16,78,12.440,10.440,5.610,5.670,1.390,2.130,60984,14,71.470,0.070
DEL007,Noroeste,32,118,21,9,44,17.670,9.670,3.000,5.200,1.840,1.950,36810,5,60.370,0.090
DEL008,Centro,45,183,23,13,60,16.670,9.500,3.210,5.360,1.520,2.040,60030,9,72.740,0.100
DEL009,Levante,38,126,19,11,44,8.500,8.000,3.400,4.130,2.160,1.680,45252,14,71.480,0.090


,medicos,visitas,ventas_extra_eur
region,,,
Centro,492,2080,560304
Levante,434,1669,606780
Noreste,460,1875,507690
Noroeste,302,1307,438588
Sur,312,1347,327546


Médicos que suben: 1130 (de ellos 507 pasan de 0 a alguna visita) | bajan: 595 | visitas que cambian de médico: 2,734 de 8,278 (33%)
Ningún delegado cambia su carga: True


## 6. Extensión: ¿y si cambia la premisa?

Las mismas funciones responden a escenarios nuevos. Dos ejemplos que suelen salir en comité:

- **¿Y si la capacidad no fuera fija?** Con el reparto óptimo, cada visita adicional vale
  hoy entre 500 y 600 € de venta anual frente a un coste de 72 €. La conclusión no es
  "recortar A", es que el esfuerzo comercial rinde mucho y está mal colocado.
- **¿Y si hubiera que recortar un 10 % o un 20 % de visitas?** Bien repartidas, se seguiría
  vendiendo más que hoy.

In [18]:
filas = []
for factor in [0.8, 0.9, 1.0, 1.1, 1.2]:
    b = int(round(presupuesto_25 * factor))
    v_b = asignacion_optima(cat, b)
    filas.append({"visitas": b, "vs_hoy": f"{factor - 1:+.0%}", "ventas_extra_M€": valor_plan(v_b) / 1e6,
                  "coste_extra_M€": (b - presupuesto_25) * COSTE_VISITA / 1e6})
sens_capacidad = pd.DataFrame(filas).set_index("visitas")
sens_capacidad["margen_extra_M€"] = sens_capacidad["ventas_extra_M€"] - sens_capacidad["coste_extra_M€"]
display(sens_capacidad.round(2))
# Valor marginal de una visita más en el óptimo (para el ejercicio de extensión)
v_mas = asignacion_optima(cat, presupuesto_25 + 500)
print("Valor medio de cada una de las 500 visitas siguientes, en el óptimo:", f"{(valor_plan(v_mas) - valor_plan(v_global)) / 500:,.0f} €")

,vs_hoy,ventas_extra_M€,coste_extra_M€,margen_extra_M€
visitas,,,,
6622,-20%,1.360,-0.120,1.480
7450,-10%,1.970,-0.060,2.030
8278,+0%,2.500,0.000,2.500
9106,+10%,2.960,0.060,2.910
9934,+20%,3.380,0.120,3.260


Valor medio de cada una de las 500 visitas siguientes, en el óptimo:

 577 €


## 7. Cifras finales

In [19]:
resumen = {
    "ventas_2025_eur": round(float(ventas[2025])),
    "cuota_global_2025": round(float(cuota_global[2025]), 4),
    "visitas_2025_validas": presupuesto_25,
    "visitas_duplicadas_eliminadas": int(dup.sum()),
    "pct_visitas_decima_o_posterior": round(visitas_10mas / presupuesto_25, 3),
    "medicos_ano_sin_visitas": int((dy.visitas == 0).sum()),
    "medicos_sin_visitas_2025": int((v_act == 0).sum()),
    "mal_segmentados": mal_segmentados,
    "visitas_por_medico_hoy": v_act.groupby(seg).mean().round(1).to_dict(),
    "visitas_por_medico_plan": v_plan.groupby(seg).mean().round(1).to_dict(),
    "regla_sencilla": regla.to_dict(),
    "curva": {"D": round(D, 4), "h": round(h, 3), "cuota_base": round(s0, 4), "techo": round(s0 + D, 4)},
    "cuota_0_visitas": round(float(dy[dy.visitas == 0].cuota.mean()), 4),
    "cuota_3_5_visitas": round(float(dy[dy.tramo == "3-5"].cuota.mean()), 4),
    "cuota_10_mas_visitas": round(float(dy[dy.visitas >= 10].cuota.mean()), 4),
    "pp_por_visita_desde_0": round(float(fd_niveles.loc["0", "pp_cuota_por_visita"]), 2),
    "pp_por_visita_desde_10_14": round(float(fd_niveles.loc["10-14", "pp_cuota_por_visita"]), 2),
    "pp_por_visita_desde_20": round(float(fd_niveles.loc["20+", "pp_cuota_por_visita"]), 2),
    "eur_visita_k_mediano": {s: {int(k): int(k_val.loc[k, s]) for k in [1, 2, 5, 10, 14, 20]} for s in "ABC"},
    "valor_plan_eur": round(valor_base),
    "valor_plan_pct_ventas": round(valor_base / float(ventas[2025]), 3),
    "valor_por_segmento_eur": {s: round(float(v)) for s, v in bridge.items()},
    "cuota_global_plan": round(float(resumen_esc.loc["Plan (óptimo por delegado)", "cuota_global"]), 4),
    "valor_optimo_global_eur": round(valor_plan(v_global)),
    "valor_regla_sencilla_eur": round(valor_plan(v_regla)),
    "valor_nadie_a_cero_eur": round(valor_plan(v_nadie0)),
    "rango_robustez_M€": [round(rango_prudente[0], 2), round(rango_prudente[1], 2)],
    "bootstrap_ic95_M€": [round(ic_boot[0] / 1e6, 2), round(ic_boot[1] / 1e6, 2)],
    "medicos_suben": suben, "medicos_bajan": bajan, "medicos_de_cero_a_visitados": de_cero,
    "visitas_que_cambian_de_medico": movidas,
    "valor_visita_extra_en_optimo_eur": round((valor_plan(v_mas) - valor_plan(v_global)) / 500),
}
(OUT / "resumen_cifras.json").write_text(json.dumps(resumen, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(resumen, indent=2, ensure_ascii=False))

# Datos detrás de cada gráfico, para reproducirlos en la presentación
datos_graficos = {
    "01_distribucion_visitas_2025": {s: dist[s].astype(int).to_dict() for s in "ABC"},
    "02_pp_cuota_por_visita_segun_nivel_2024": fd_niveles.pp_cuota_por_visita.round(2).to_dict(),
    "03_curva": {
        "observado_por_segmento": {s: curva_obs.loc[s][["visitas", "cuota", "n"]].round(3).to_dict("index") for s in "ABC"},
        "curva_valoracion_pct": {int(v): round(100 * (s0 + g(v, D, h)), 1) for v in range(0, 31)},
        "curva_niveles_pct": {int(v): round(100 * (res_niv.x[0] + g(v, *ajustes["Niveles (todos)"])), 1) for v in range(0, 31)},
        "visitas_hoy_por_segmento": v_act.groupby(seg).mean().round(1).to_dict(),
    },
    "04_eur_visita_k_medico_mediano": {s: {int(k): int(v) for k, v in k_val[s].items()} for s in "ABC"},
    "05_puente_M€": {"perdida_en_medicos_que_bajan": round(perdida / 1e6, 2), "n_medicos_bajan": int(n_bajan.sum()),
                     **{f"ganancia_{s}": round(float(ganancia[s]) / 1e6, 2) for s in "ABC"},
                     **{f"n_medicos_suben_{s}": int((v_plan > v_act)[seg == s].sum()) for s in "ABC"}, "neto": round(valor_base / 1e6, 2)},
    "06_frecuencia_hoy_vs_plan": resumen_plan[["hoy", "plan", "plan_min", "plan_max"]].round(1).to_dict("index"),
    "escenarios": resumen_esc.round(3).to_dict("index"),
    "robustez_M€": robustez["ventas_extra_M€"].round(2).to_dict(),
    "sensibilidad_capacidad": sens_capacidad.round(2).to_dict("index"),
    "plan_por_region_eur": plan_reg.to_dict("index"),
    "efecto_mensual_pp": efecto_mensual.pp_cuota.to_dict(),
}
(OUT / "datos_graficos.json").write_text(json.dumps(datos_graficos, indent=2, ensure_ascii=False), encoding="utf-8")

{
  "ventas_2025_eur": 30605130,
  "cuota_global_2025": 0.1992,
  "visitas_2025_validas": 8278,
  "visitas_duplicadas_eliminadas": 255,
  "pct_visitas_decima_o_posterior": 0.225,
  "medicos_ano_sin_visitas": 1006,
  "medicos_sin_visitas_2025": 509,
  "mal_segmentados": 30,
  "visitas_por_medico_hoy": {
    "A": 13.3,
    "B": 4.1,
    "C": 1.5
  },
  "visitas_por_medico_plan": {
    "A": 9.6,
    "B": 5.0,
    "C": 1.9
  },
  "regla_sencilla": {
    "A": 9,
    "B": 5,
    "C": 2
  },
  "curva": {
    "D": 0.159,
    "h": 2.536,
    "cuota_base": 0.1172,
    "techo": 0.2762
  },
  "cuota_0_visitas": 0.1238,
  "cuota_3_5_visitas": 0.2104,
  "cuota_10_mas_visitas": 0.2352,
  "pp_por_visita_desde_0": 3.65,
  "pp_por_visita_desde_10_14": 0.14,
  "pp_por_visita_desde_20": 0.05,
  "eur_visita_k_mediano": {
    "A": {
      "1": 9881,
      "2": 5524,
      "5": 1799,
      "10": 613,
      "14": 345,
      "20": 183
    },
    "B": {
      "1": 3833,
      "2": 2143,
      "5": 698,
      "1

8456

## 8. Límites y supuestos

- **Identificación.** La curva se estima con cambios dentro del mismo médico entre 2024 y
  2025, lo que elimina diferencias fijas entre médicos. No elimina un posible sesgo por
  tendencias: si los delegados empezaron a visitar a médicos cuya cuota ya subía, el efecto
  estaría algo sobreestimado. Los 153 médicos que pasaron de 0 visitas en 2024 a alguna en
  2025 muestran una cuota algo superior a la media de no visitados ya antes de la primera
  visita, así que el sesgo existe y es moderado; por eso se da un rango y no un solo número.
- **Misma curva para todos.** Se asume que la ganancia de cuota por visita depende del número
  de visitas y no del segmento; los ajustes por segmento son muy parecidos (tabla 2.4) y la
  valoración con curvas separadas cambia el valor en menos de 0,3 M€.
- **Potencial estable.** El volumen de categoría de cada médico cambia poco entre años
  (correlación 0,998), así que 2025 sirve como base para 2026.
- **Efecto simétrico.** Se asume que quitar visitas a un A le baja la cuota tanto como
  dárselas se la sube. Los datos lo respaldan (sección 2.2) y ese coste está descontado en
  el valor del plan.
- **Sin efectos cruzados ni de competencia.** No hay datos de la actividad de competidores ni
  de reacción a la nuestra; el periodo no tiene lanzamientos ni cambios de precio.
- **Coste.** El plan mantiene las visitas de cada delegado, así que el coste total no cambia.
  Las diferencias de coste por delegado (59 a 91 €) no se usan porque no se mueven visitas
  entre delegados.
- **Datos.** 255 visitas duplicadas eliminadas (sensibilidad en 4.1); las fechas se reparten
  por igual entre los siete días de la semana, así que no se hace ningún análisis por día;
  los códigos de brick se repiten entre Noreste y Noroeste (no se usan bricks en el análisis).